# 13. Baseline ML và MLP trên NF-BoT-IoT-v3 (task 4.1, GĐ D)

Đọc đồ thị từ output của kernel `gnn-iot-ids-11-build-graphs-bot` (`processed/nf_bot_iot_v3/`). Dùng đúng `split.json` của kernel đó, không chia lại.
Chạy `scripts/run_baselines.py`: với mỗi mô hình (logreg, random_forest, xgboost, lightgbm, mlp) chọn cấu hình trên **val** (seed 42, 2-3 lần thử mỗi mô hình, có log), rồi chạy 3 seed 42, 43, 44; chỉ ghi `split=val` (test chỉ dùng một lần ở GĐ H).

**NF-BoT-IoT-v3 chỉ có 3 lớp**: Normal (0), DoS_DDoS (1), Truy_cap_trai_phep (2); không có Botnet_Backdoor và Khac. Số lớp lấy từ `split.json["classes"]`; F1-macro chỉ tính trên các lớp có mặt trong nhãn thật. `fpr`, `recall_attack`, `roc_auc` là chỉ số nhị phân (Normal là lớp âm) suy ra từ dự đoán đa lớp.

Kaggle CPU (không bật GPU), bật Internet. Ô cuối chỉ ĐẾM số dòng theo (model, seed); không tính trung bình (tổng hợp làm ở GĐ H).

In [ ]:
import csv, glob, json, os, resource, subprocess, sys, time
from collections import Counter
from contextlib import contextmanager

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"          # ngoài /kaggle/working để không lẫn vào output
INFO = {"steps_s": {}, "device": "CPU", "gpu": None}
T_START = time.time()


def peak_ram_gb():
    """RAM đỉnh (GB) của tiến trình notebook và mọi tiến trình con đã kết thúc (ru_maxrss tính bằng KB trên Linux)."""
    kb = max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss, resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss)
    return round(kb / 2**20, 2)


def save_info():
    INFO["total_s"] = round(time.time() - T_START, 1)
    INFO["peak_ram_gb"] = peak_ram_gb()
    with open(f"{WORK}/run_info.json", "w", encoding="utf-8") as f:
        json.dump(INFO, f, ensure_ascii=False, indent=1)


@contextmanager
def step(name):
    t0 = time.time()
    try:
        yield
    finally:
        INFO["steps_s"][name] = round(time.time() - t0, 1)
        save_info()                # ghi sau mỗi bước, lỗi giữa chừng vẫn còn run_info.json
        print(f"--- {name}: {INFO['steps_s'][name]} s, RAM đỉnh tới nay {INFO['peak_ram_gb']} GB")


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-9000:], end="")
    print(r.stderr[-2500:], end="", file=sys.stderr)
    print(f"[mã thoát {r.returncode}]")
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


with step("clone_va_cai_goi"):
    run(["rm", "-rf", REPO])
    run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
    run(["git", "-C", REPO, "log", "-1", "--format=%H %s"])
    INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    with open(f"{REPO}/requirements.txt") as f:
        reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
    keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
    run([sys.executable, "-m", "pip", "install", "-q", *keep])

try:
    import torch
    if torch.cuda.is_available():
        INFO["device"], INFO["gpu"] = "GPU", torch.cuda.get_device_name(0)
except ImportError:
    pass
INFO["cpu_count"] = os.cpu_count()
print("Commit:", INFO["commit"], "| máy:", INFO["device"], INFO["gpu"], "| CPU:", INFO["cpu_count"])

## 1. pytest (dừng nếu có test lỗi)

In [ ]:
with step("pytest"):
    r = run([sys.executable, "-m", "pytest", "-q"], cwd=REPO)      # mã thoát khác 0 thì run() ném lỗi, notebook dừng
    INFO["pytest"] = r.stdout.strip().splitlines()[-1]

## 2. Baseline trên NF-BoT-IoT-v3

In [ ]:
DS = "nf_bot_iot_v3"
D = os.path.dirname(next(glob.iglob(f"/kaggle/input/**/{DS}/split.json", recursive=True)))
print("data_dir:", D)
meta = json.load(open(f"{D}/split.json", encoding="utf-8"))
print("Lớp theo split.json:", meta["classes"], "| số node:", {p: meta[p].get("n_nodes") for p in ("train", "val", "test")})
INFO["classes"] = meta["classes"]
RESULTS = f"{WORK}/results_baselines_{DS}.csv"
SEARCH = f"{WORK}/baseline_search_{DS}.json"
for p in (RESULTS, SEARCH):
    if os.path.exists(p):
        os.remove(p)               # chạy lại từ đầu thì không nối thêm dòng trùng

with step("run_baselines"):
    run([sys.executable, "-I", "scripts/run_baselines.py", D, DS, RESULTS, SEARCH], cwd=REPO)

## 3. Đếm số dòng theo (model, seed) và số lần thử cấu hình (không tính trung bình)

In [ ]:
with open(RESULTS, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
cnt = Counter((r["model"], int(r["seed"]), r["task"], r["split"]) for r in rows)
print(f"{len(rows)} dòng; datasets: {sorted({r['dataset'] for r in rows})}; split: {sorted({r['split'] for r in rows})}; task: {sorted({r['task'] for r in rows})}")
for (m, s, t, sp), n in sorted(cnt.items()):
    print(f"  {m:14s} seed {s} {t} {sp}: {n} dòng")
search = json.load(open(SEARCH, encoding="utf-8"))
tries = {m: sum(1 for e in v if "cfg" in e) for m, v in search.items()}
print("Số lần thử cấu hình trên val:", tries)
want = {(m, s) for m in ("logreg", "random_forest", "xgboost", "lightgbm", "mlp") for s in (42, 43, 44)}
INFO["rows"] = len(rows)
INFO["rows_by_model_seed"] = {f"{m}|{s}": n for (m, s, _, _), n in sorted(cnt.items())}
INFO["config_tries"] = tries
INFO["complete_5x3"] = {(m, s) for (m, s, _, _) in cnt} == want and all(n == 1 for n in cnt.values())
INFO["only_val"] = {r["split"] for r in rows} == {"val"}
INFO["max_tries_ok"] = all(n <= 30 for n in tries.values())
save_info()
print("\nrun_info.json:")
print(open(f"{WORK}/run_info.json", encoding="utf-8").read())
assert INFO["complete_5x3"] and INFO["only_val"] and INFO["max_tries_ok"], "Kết quả chưa đạt tiêu chí 4.1, xem số đếm ở trên"